# Beat the Bytes
**MassAI · November 18, 2026 · 40-minute hands-on lab**

How few bytes can your heartbeat classifier use while still catching PVCs? Work in teams of 2–3. Compare techniques and share what you learned.

**Start here** · Sign in to Colab, use a CPU runtime, and run setup. The public workshop data and pretrained model download automatically. Run the cells in order. Edit only the marked experiment settings in Stage C and your team name and trial number in Stage D.

If a download fails, rerun setup. A presenter can give you the student kit for the manual upload option. For local Jupyter, keep `model-golf-data.zip` beside the notebook.

In [ ]:
#@title Setup — load the workshop files
from pathlib import Path
from zipfile import ZipFile
from urllib.request import urlopen
from hashlib import sha256
import sys

manual_upload = False  # Set True only if the automatic download fails.
archive_name = 'model-golf-data.zip'
source = 'https://raw.githubusercontent.com/bvetsa/MassAI_Harware_Workshop/7241b01e74d104842c6068d1a73c60e8fcce164d/'
checksums = {'classroom.npz': '658019ee9239fabe8ed9a491ec397f33421e5d1e3537d79889c46f1fae7a6681', 'baseline.mgf': 'f01ba0e654aa7c44e35d94ba8309807d90277f7949c71060a6af7ed200f5788f', 'model_golf.py': '1250c872a6e2f9f3a8708672cba0d62e036be6595415eabe55e8f54eba420107', 'rules.json': 'da534554b4dd8e344e8985c3b5d0b7a88b3ecc2ce0b388b232735b0ca31ec6a6'}
base = Path('.')
if not all((base / name).is_file() for name in checksums):
    base = Path('model-golf-data')
    if manual_upload and not Path(archive_name).is_file():
        try:
            from google.colab import files
        except ImportError as error:
            raise FileNotFoundError(f'Put {archive_name} beside this notebook, then rerun setup.') from error
        print(f'Choose {archive_name} from the student kit.')
        uploaded = files.upload()
        if archive_name not in uploaded:
            raise ValueError(f'Please choose {archive_name}.')
        Path(archive_name).write_bytes(uploaded[archive_name])
    for name, digest in checksums.items():
        target = base / name
        if target.is_file() and sha256(target.read_bytes()).hexdigest() == digest:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        if Path(archive_name).is_file():
            with ZipFile(archive_name) as archive:
                content = archive.read(name)
        else:
            try:
                with urlopen(source + name, timeout=30) as response:
                    content = response.read()
            except OSError as error:
                raise RuntimeError('Download failed. Rerun setup. If it keeps failing, get the student kit '
                                   'from a presenter, set manual_upload = True, and rerun setup.') from error
        if sha256(content).hexdigest() != digest:
            raise ValueError(f'{name} differs from the workshop version. Use the current student kit.')
        target.write_bytes(content)

import numpy as np
sys.path.insert(0, str(base.resolve()))
from model_golf import (BASELINE_WIDTH, INPUTS, SEED, TRAINING_EPOCHS, TRAINING_SEEDS,
                        WIDTHS, LABELS, decode, encode, format_score, score, train,
                        browser_sketch, firmware_headers, logits)

with np.load(base / 'classroom.npz', allow_pickle=False) as data:
    x_train, y_train = data['train_x'], data['train_y']
    x_val, y_val = data['validation_x'], data['validation_y']
baseline = (base / 'baseline.mgf').read_bytes()
print(f'Ready · {len(x_train):,} training windows · {len(x_val):,} practice windows · 64 values per beat')


## A · Baseline — 5 minutes

Each input is a window around an annotated heartbeat from MIT-BIH. The model distinguishes **N** (normal beat) from **V** (premature ventricular contraction, or PVC). It reads 64 ECG values, combines them in one hidden layer, and outputs a score. A score of zero or higher predicts V.

**Accuracy** is the share of all beats classified correctly. **PVC sensitivity** is the share of annotated PVC beats the model catches. Normal beats outnumber PVCs, so guessing N every time can score well on accuracy while missing every PVC.

Load the pretrained starter model below. Training a smaller model comes later in Stage C. **False negatives** are missed PVCs. **False positives** are normal beats flagged as PVC. Both matter, so read the confusion counts together with accuracy and sensitivity.

These are preprocessed windows from two expert beat labels. The dial demonstration selects recorded examples. Clinical evaluation would require a separate study.

In [ ]:
import json

rules = json.loads((base / 'rules.json').read_text())
accuracy_floor = rules['minimum_accuracy']
sensitivity_floor = rules['minimum_pvc_sensitivity']
baseline_model = decode(baseline)
parameters = baseline_model.w1.size + baseline_model.b1.size + baseline_model.w2.size + 1
baseline_score = score(baseline, x_val, y_val)

print(f'{INPUTS} ECG values → {BASELINE_WIDTH} ReLU neurons → 1 score')
print(f'{parameters:,} learned parameters · FP32 weights')
print(f"Practice: {baseline_score['bytes']:,} bytes · {baseline_score['accuracy']:.2%} accuracy · "
      f"{baseline_score['sensitivity']:.2%} PVC sensitivity")
print(f"Correct N: {baseline_score['tn']:,}  |  N flagged as PVC: {baseline_score['fp']:,}")
print(f"Missed PVC: {baseline_score['fn']:,}  |  Correct PVC: {baseline_score['tp']:,}")
print(f'Always predict N: {np.mean(y_val == 0):.1%} accuracy, 0% PVC sensitivity')
print(f'Lab practice floors: {accuracy_floor:.0%} accuracy and {sensitivity_floor:.0%} PVC sensitivity')

## B · First optimization — 5 minutes

Start with the same trained model. A 32-bit floating-point weight takes four bytes. INT8 stores a rounded weight in one byte and adds a scale for each layer. The simulator uses those scales to recover approximate weights and still accumulates in FP32. The model keeps the same number of neurons and parameters.

Before running the cell, guess how many bytes you’ll save. Compare the size and both practice metrics after the conversion. This exercise uses pretrained weights and runs without training.

The bytes below include the model’s header, weights, biases and INT8 scales. The complete ESP32 program also stores inference code and example inputs and uses RAM to run.

In [ ]:
int8_baseline = encode(baseline_model, bits=8)
int8_score = score(int8_baseline, x_val, y_val)
saved = len(baseline) - len(int8_baseline)

print('FP32 before:', format_score(baseline_score))
print('INT8 after: ', format_score(int8_score))
print(f'Saved {saved:,} bytes ({saved / len(baseline):.1%}) with {parameters:,} parameters in both models.')

## C · Model Golf — 15 minutes

Edit the trial settings and rerun that cell. Try **16**, **8**, then **4** hidden neurons with INT8. Fewer neurons mean fewer weights and biases. Change `bits` to **32** on one of your widths to isolate what quantization changed.

| Setting | Choices | What it changes |
|---|---|---|
| `width` | `4`, `8`, `16`, `24` | Number of neurons in the hidden layer |
| `bits` | `8`, `32` | Precision of stored weights |
| `seed` | `18`, `7`, `42` | Initial weights, sampled normal beats, and training order |
| `training` | `"standard"`, `"short"` | 45 or 15 passes through the training subset |

Leave `seed` and `training` at their defaults for your first trials. **Optional challenge:** At the same width and precision, change just the seed or training length. Can you catch more PVCs for the same number of bytes? Use the same settings to repeat a trial. Scores can vary slightly across platforms.

Numbers in the table use the **practice recordings**. A pass on these practice recordings does not guarantee the same result on a new recording. The plot’s dashed line marks the sensitivity floor. Point color shows whether the trial meets **both** practice floors. Run the next cell once to start your trial history.

**Edit only the four settings at the top of the trial cell.** Rerun that cell after each change. The helper cell records your history and draws the chart. Each plotted point is one tested model. If points overlap, use the numbered table to compare them. The outlined point is your smallest passing model so far.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Reuse one trained model for both weight formats. Keep scores by trial number.
fitted = {(BASELINE_WIDTH, SEED, 'standard'): baseline_model}
trials = {}


def record_trial(key, payload, metrics):
    passes = metrics['accuracy'] >= accuracy_floor and metrics['sensitivity'] >= sensitivity_floor
    trials[key] = dict(metrics, number=len(trials) + 1, payload=payload, passes=passes)


def run_trial(width, bits, seed, training):
    if type(width) is not int or type(bits) is not int or width not in WIDTHS or bits not in (8, 32):
        raise ValueError('Pick one of the listed widths and use 8 or 32 bits.')
    if seed not in TRAINING_SEEDS or training not in TRAINING_EPOCHS:
        raise ValueError('Pick a listed seed and either "standard" or "short" training.')
    key = (width, bits, seed, training)
    if key not in trials:
        fit_key = (width, seed, training)
        if fit_key not in fitted:
            fitted[fit_key] = train(x_train, y_train, width, seed=seed, training=training)
        payload = encode(fitted[fit_key], bits)
        record_trial(key, payload, score(payload, x_val, y_val))
    return trials[key]


def smallest_passing_trial():
    passing = (row for row in trials.values() if row['passes'])
    return min(passing, key=lambda row: (row['bytes'], -row['sensitivity'], -row['accuracy']), default=None)


def show_trials():
    best = smallest_passing_trial()
    print('Trial  Architecture  Weights  Training / seed  Bytes  Accuracy  PVC sensitivity  Practice floors')
    for (width, bits, seed, training), row in trials.items():
        marker = '*' if row is best else ' '
        architecture = f'{INPUTS}-{width}-1'
        precision = 'INT8' if bits == 8 else 'FP32'
        configuration = f'{training} / {seed}'
        status = 'Pass' if row['passes'] else 'Below floor'
        print(f"{marker}{row['number']:>3}   {architecture:<12}  {precision:<7}  {configuration:<15}"
              f" {row['bytes']:>5,}  {row['accuracy']:>8.2%}  {row['sensitivity']:>15.2%}  {status}")

    fig, ax = plt.subplots(figsize=(8, 3.7), layout='constrained')
    for passed, color, marker, label in ((True, '#157467', 'o', 'Meets both practice floors'),
                                         (False, '#aa4e26', 'x', 'Below a practice floor')):
        group = [row for row in trials.values() if row['passes'] == passed]
        if group:
            ax.scatter([row['bytes'] for row in group], [row['sensitivity'] for row in group],
                       color=color, marker=marker, s=44, label=label, zorder=3)
    ax.axhline(sensitivity_floor, color='#4b5563', linestyle='--',
               label=f'{sensitivity_floor:.0%} PVC sensitivity floor')
    if best is not None:
        ax.scatter(best['bytes'], best['sensitivity'], s=170, facecolors='none',
                   edgecolors='#102e48', linewidths=1.6, label=f"Smallest passing trial #{best['number']}", zorder=4)
        print(f"* Smallest model meeting both practice floors: trial #{best['number']} ({best['bytes']:,} bytes).")
    else:
        print('No trial has met both practice floors yet.')
    ax.set(xlabel='Exported model size (bytes)', ylabel='Practice PVC sensitivity', xlim=(0, None))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y', alpha=0.2)
    ax.legend(loc='best')
    plt.show()


record_trial((BASELINE_WIDTH, 32, SEED, 'standard'), baseline, baseline_score)
record_trial((BASELINE_WIDTH, 8, SEED, 'standard'), int8_baseline, int8_score)
print('Trials #1 and #2 are your FP32 and INT8 baseline models.')

In [ ]:
# Edit these settings, then run this cell again.
width = 16
bits = 8
seed = 18
training = 'standard'

run_trial(width, bits, seed, training)
show_trials()

## D · Export and share — 5 minutes

Pick a trial number from your table and enter it with your team name below. The cell validates your choice, prints its configuration and practice metrics, and downloads a `.mgf` file. In local Jupyter, get the file from the notebook folder. Export is useful even if your smallest trial is below a floor.

For voluntary show-and-tell, share your byte count, accuracy, PVC sensitivity and the change that helped most. Aim for at least **80% accuracy** and **94% PVC sensitivity** on the practice recordings, then find the smallest model that meets both. Tell the room about a tradeoff you saw. A presenter may deploy a volunteered model in the finale.

In [ ]:
import re

team_name = ''
trial_number = None  # Replace with a trial number from your table.


def export_submission(team, number):
    if not isinstance(team, str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9 _-]{0,39}', team.strip()):
        raise ValueError('Use 1–40 letters, digits, spaces, underscores, or hyphens. Start with a letter or digit.')
    by_number = {row['number']: row for row in trials.values()}
    if type(number) is not int or number not in by_number:
        raise ValueError('Choose a whole trial number from the results table.')
    trial = by_number[number]
    decode(trial['payload'])  # Confirm the chosen file is a valid ESP32 model before writing it.
    width, bits, seed, training = next(key for key, row in trials.items() if row is trial)
    path = Path(f'{team.strip()}.mgf')
    path.write_bytes(trial['payload'])
    print(f"Trial #{number} → {path.name} ({path.stat().st_size:,} bytes)")
    print(f"Architecture 64-{width}-1 · {'INT8' if bits == 8 else 'FP32'} weights · seed {seed} · {training} training")
    print(f"Practice: {trial['accuracy']:.2%} accuracy · {trial['sensitivity']:.2%} PVC sensitivity")
    return path


if team_name or trial_number is not None:
    submission = export_submission(team_name, trial_number)
    try:
        from google.colab import files
    except ImportError:
        print(submission.resolve())
    else:
        files.download(str(submission))
else:
    print('Enter your team name and a trial number, then rerun this cell.')

## Share what you learned — 5 minutes

What saved the most bytes? Which model missed the fewest PVCs? What happened when you changed just one setting? How might your choice change on a new recording?

**Troubleshooting**

- If a name is undefined, run setup and then the cells above it in order.
- If your trial is below a floor, try more neurons or standard training, then compare both metrics.
- If setup fails to download, rerun it or ask a presenter for the manual data ZIP.
- Rerunning the trial helper cell clears the history. Run it once at the start of Stage C.
- Use a trial number from the latest table when exporting. A filename starts with a letter or digit.

[MIT-BIH Arrhythmia Database](https://physionet.org/content/mitdb/1.0.0/) · [WFDB beat annotation codes](https://physionet.org/physiotools/wpg/wpg_36.htm)